In [1]:
import pandas as pd
import numpy as np
import re

import nltk

from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

I0000 00:00:1787761784.533974  139990 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1787761784.629396  139990 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787761786.668764  139990 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
nltk.download("stopwords")

/home/aximsoft/snap/code/258/.local/share/virtualenvs/Text_Sentiment_Analysis-D79r_KZq/lib/python3.13/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/aximsoft/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/aximsoft/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
df = pd.read_csv("../Dataset/IMDB Dataset.csv")

In [4]:
df.shape

(50000, 2)

In [5]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [6]:
print(df.isnull().sum())

review       0
sentiment    0
dtype: int64


In [7]:
df.duplicated().sum()

np.int64(418)

In [8]:
df = df.drop_duplicates().reset_index(drop=True)

In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
df.shape

(49582, 2)

# Encode

In [11]:
sentiment_mapping = {
    "negative": 0,
    "positive": 1
}

df["sentiment_label"] = df["sentiment"].map(sentiment_mapping)

df[["sentiment", "sentiment_label"]].head()

,sentiment,sentiment_label
0,positive,1
1,positive,1
2,positive,1
3,negative,0
4,positive,1


In [12]:
print(df["sentiment_label"].value_counts())

sentiment_label
1    24884
0    24698
Name: count, dtype: int64


# Create the Text Cleaning Function

In [13]:
# NEGATION_WORDS = {
#     "no",
#     "not",
#     "nor",
#     "never",
#     "neither",
#     "none",
#     "nobody",
#     "nothing",
#     "nowhere",
#     "hardly",
#     "scarcely",
#     "barely",
#     "isn't",
#     "wasn't",
#     "didn't",
#     "don't",
#     "can't",
#     "couldn't",
#     "won't"
# }

stop_words = set(stopwords.words("english"))

# stop_words = stop_words - NEGATION_WORDS

def clean_text(text):

    text = text.lower()

    text = re.sub(r"<.*?>", " ", text)

    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    words = text.split()

    words = [
        word
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

# test the clean function

In [14]:
sample_review = df.loc[0, "review"]

print("Original review:\n")
print(sample_review)

print("\n" + "=" * 80)

cleaned_review = clean_text(sample_review)

print("Cleaned review:\n")
print(cleaned_review)

Original review:

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show

In [15]:
# Apply Cleaning to the Complete Dataset

df["cleaned_review"] = df["review"].apply(clean_text)

In [16]:
df[["review", "cleaned_review", "sentiment"]].head()

,review,cleaned_review,sentiment
0,One of the other reviewers has mentioned that ...,one reviewers mentioned watching oz episode ho...,positive
1,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...,positive
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...,positive
3,Basically there's a family where a little boy ...,basically family little boy jake thinks zombie...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visually stunnin...,positive


# Check for Empty Reviews

In [17]:
empty_reviews = (df["cleaned_review"].str.strip() == "").sum()

print("Empty reviews after preprocessing:", empty_reviews)

Empty reviews after preprocessing: 0


In [18]:
df.shape

(49582, 4)

In [19]:
df.head()

,review,sentiment,sentiment_label,cleaned_review
0,One of the other reviewers has mentioned that ...,positive,1,one reviewers mentioned watching oz episode ho...
1,A wonderful little production. <br /><br />The...,positive,1,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,positive,1,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,negative,0,basically family little boy jake thinks zombie...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1,petter mattei love time money visually stunnin...


# Input and output 

In [20]:
X = df["cleaned_review"]

y = df["sentiment_label"]

In [21]:
print("Number of reviews:", len(X))
print("Number of labels:", len(y))

Number of reviews: 49582
Number of labels: 49582


In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

In [23]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.1765,
    random_state=42,
    stratify=y_train
)

In [24]:
print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))

Training samples: 34705
Validation samples: 7439
Testing samples: 7438


# Verify Class Distribution

In [25]:
print("Training distribution:")
print(y_train.value_counts(normalize=True))

print("\nValidation distribution:")
print(y_val.value_counts(normalize=True))

print("\nTest distribution:")
print(y_test.value_counts(normalize=True))

Training distribution:
sentiment_label
1    0.501887
0    0.498113
Name: proportion, dtype: float64

Validation distribution:
sentiment_label
1    0.501815
0    0.498185
Name: proportion, dtype: float64

Test distribution:
sentiment_label
1    0.501882
0    0.498118
Name: proportion, dtype: float64


# Tokenization

In [26]:
tokenizer = Tokenizer(
    num_words=20000,
    oov_token="<OOV>"
)

In [27]:
tokenizer.fit_on_texts(X_train)

In [28]:
# Check vocab   

tokenizer.word_index

{'<OOV>': 1,
 'movie': 2,
 'film': 3,
 'one': 4,
 'like': 5,
 'good': 6,
 'time': 7,
 'even': 8,
 'would': 9,
 'story': 10,
 'really': 11,
 'see': 12,
 'well': 13,
 'much': 14,
 'bad': 15,
 'get': 16,
 'great': 17,
 'people': 18,
 'also': 19,
 'first': 20,
 'made': 21,
 'make': 22,
 'could': 23,
 'way': 24,
 'movies': 25,
 'think': 26,
 'characters': 27,
 'character': 28,
 'watch': 29,
 'films': 30,
 'two': 31,
 'seen': 32,
 'many': 33,
 'love': 34,
 'acting': 35,
 'plot': 36,
 'never': 37,
 'life': 38,
 'best': 39,
 'show': 40,
 'know': 41,
 'little': 42,
 'ever': 43,
 'man': 44,
 'better': 45,
 'end': 46,
 'scene': 47,
 'still': 48,
 'say': 49,
 'scenes': 50,
 'something': 51,
 'go': 52,
 'back': 53,
 'real': 54,
 'thing': 55,
 'watching': 56,
 'actors': 57,
 'years': 58,
 'though': 59,
 'director': 60,
 'funny': 61,
 'another': 62,
 'old': 63,
 'actually': 64,
 'work': 65,
 'makes': 66,
 'nothing': 67,
 'look': 68,
 'going': 69,
 'find': 70,
 'lot': 71,
 'new': 72,
 'every': 73,
 'p

In [29]:
word_index = tokenizer.word_index

print("Total vocabulary:", len(word_index))

Total vocabulary: 85869


In [30]:
list(word_index.items())[:20]

[('<OOV>', 1),
 ('movie', 2),
 ('film', 3),
 ('one', 4),
 ('like', 5),
 ('good', 6),
 ('time', 7),
 ('even', 8),
 ('would', 9),
 ('story', 10),
 ('really', 11),
 ('see', 12),
 ('well', 13),
 ('much', 14),
 ('bad', 15),
 ('get', 16),
 ('great', 17),
 ('people', 18),
 ('also', 19),
 ('first', 20)]

In [31]:
# Training:

X_train_sequences = tokenizer.texts_to_sequences(X_train)

# `Validation:

X_val_sequences = tokenizer.texts_to_sequences(X_val)

# Testing:

X_test_sequences = tokenizer.texts_to_sequences(X_test)

In [32]:
print("Original text:")
print(X_train.iloc[0])

print("\nInteger sequence:")
print(X_train_sequences[0])

Original text:
much say one except probably worst early spate zombie movies may get watch another one revolt zombies month star john carradine intention building army service third reich seen much james baskett uncle remus song south plays leader also serves carradine manservant black comic mantan moreland reprises fraidy cat chauffeur role king zombies exotically named madame sul te wan carradine housekeeper unfortunately carradine supreme achievement zombification wife brings sorts trouble relatives turn remote abode lab inquire sudden death means fake funeral service actually proves disobedient indignant eventually persuading fellow zombies rise master also involved cowboy star bob steele still best known bit howard hawks big sleep plays u secret agent posing nazi posing sheriff thankfully director sekely would much better luck next genre effort day triffids

Integer sequence:
[14, 49, 4, 430, 128, 142, 289, 1, 823, 25, 94, 16, 29, 62, 4, 7911, 1116, 3148, 204, 191, 4360, 3438, 1178

# Determine Sequence Length

In [33]:
sequence_lengths = [
    len(sequence)
    for sequence in X_train_sequences
]

print("Minimum length:", min(sequence_lengths))
print("Maximum length:", max(sequence_lengths))
print("Average length:", np.mean(sequence_lengths))
print("Median length:", np.median(sequence_lengths))

Minimum length: 3
Maximum length: 1154
Average length: 118.55683619075062
Median length: 88.0


In [34]:
print(
    "90th percentile:",
    np.percentile(sequence_lengths, 90)
)

print(
    "95th percentile:",
    np.percentile(sequence_lengths, 95)
)

90th percentile: 233.0
95th percentile: 307.0


# Padding

In [35]:
MAX_SEQUENCE_LENGTH = 200

# Training

X_train_padded = pad_sequences(
    X_train_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

# Validation:

X_val_padded = pad_sequences(
    X_val_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

# Testing:

X_test_padded = pad_sequences(
    X_test_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

In [36]:
print("X_train shape:", X_train_padded.shape)
print("X_val shape:", X_val_padded.shape)
print("X_test shape:", X_test_padded.shape)

X_train shape: (34705, 200)
X_val shape: (7439, 200)
X_test shape: (7438, 200)


In [37]:
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)
print("y_test shape:", y_test.shape)

y_train shape: (34705,)
y_val shape: (7439,)
y_test shape: (7438,)


In [38]:
y_train = np.array(y_train)
y_val = np.array(y_val)
y_test = np.array(y_test)


print(type(X_train_padded))
print(type(y_train))

print(X_train_padded.dtype)
print(y_train.dtype)

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
int32
int64


In [39]:
sample_index = 0

print("Original review:")
print(X_train.iloc[sample_index])

print("\nCleaned review:")
print(X_train.iloc[sample_index])

print("\nToken sequence:")
print(X_train_sequences[sample_index])

print("\nPadded sequence:")
print(X_train_padded[sample_index])

print("\nSentiment label:")
print(y_train[sample_index])

Original review:
much say one except probably worst early spate zombie movies may get watch another one revolt zombies month star john carradine intention building army service third reich seen much james baskett uncle remus song south plays leader also serves carradine manservant black comic mantan moreland reprises fraidy cat chauffeur role king zombies exotically named madame sul te wan carradine housekeeper unfortunately carradine supreme achievement zombification wife brings sorts trouble relatives turn remote abode lab inquire sudden death means fake funeral service actually proves disobedient indignant eventually persuading fellow zombies rise master also involved cowboy star bob steele still best known bit howard hawks big sleep plays u secret agent posing nazi posing sheriff thankfully director sekely would much better luck next genre effort day triffids

Cleaned review:
much say one except probably worst early spate zombie movies may get watch another one revolt zombies month

In [40]:
import pickle

with open(
    "../Dataset/processed/tokenizer.pkl",
    "wb"
) as file:

    pickle.dump(tokenizer, file)

In [41]:
text_splits = {
    "X_train": X_train.tolist(),
    "X_val": X_val.tolist(),
    "X_test": X_test.tolist()
}

with open(
    "../Dataset/processed/text_splits.pkl",
    "wb"
) as file:
    pickle.dump(text_splits, file)

In [42]:
preprocessing_config = {
    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "num_words": 20000
}

with open(
    "../Dataset/processed/preprocessing_config.pkl",
    "wb"
) as file:

    pickle.dump(preprocessing_config, file)

In [43]:
np.save(
    "../Dataset/processed/X_train_padded.npy",
    X_train_padded
)

np.save(
    "../Dataset/processed/X_val_padded.npy",
    X_val_padded
)

np.save(
    "../Dataset/processed/X_test_padded.npy",
    X_test_padded
)

np.save(
    "../Dataset/processed/y_train.npy",
    y_train
)

np.save(
    "../Dataset/processed/y_val.npy",
    y_val
)

np.save(
    "../Dataset/processed/y_test.npy",
    y_test
)